# LJ Dev Commerce - Phase 4

# XI. Return ETL


# =========================================================
# 1.1 Setup
# =========================================================

This notebook performs the approved ETL workflow for the Return transaction dataset.

Workflow: Raw Dataset → Clean CSV → Database-Ready Dataset → PostgreSQL.

The raw Return dataset is never modified directly.


In [1]:
# =========================================================
# 1.1 Setup
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from decimal import Decimal

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
# Project path handling is configured in the dataset load cell.


## 1.2 Load and Preserve Return Raw Dataset

Load the untouched Return transaction source dataset and preserve an original copy for validation.


In [2]:
# =========================================================
# 1.2 Load and Preserve Return Raw Dataset
# =========================================================

return_raw_path = Path(
    r"C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce"
    r"\data\11_Returns_System\returns_transactions.csv"
)

return_raw = pd.read_csv(return_raw_path)
return_raw_original = return_raw.copy(deep=True)

print("Return raw dataset loaded successfully.")
print("Rows:", len(return_raw))
print("Columns:", len(return_raw.columns))
print("Path:", return_raw_path)


Return raw dataset loaded successfully.
Rows: 4
Columns: 13
Path: C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\11_Returns_System\returns_transactions.csv


## 1.3 Initial Raw Data Inspection

Inspect the untouched Return transaction dataset before any cleaning or transformation.


In [3]:
# =========================================================
# 1.3 Initial Raw Data Inspection
# =========================================================

print("Return Dataset shape:")
print(return_raw.shape)
print("\nReturn Column names:")
print(return_raw.columns.tolist())
print("\nReturn Current pandas data types:")
print(return_raw.dtypes)
print("\nReturn Missing values:")
print(return_raw.isnull().sum())
print("\nReturn Sample raw records:")
display(return_raw.head())


Return Dataset shape:
(4, 13)

Return Column names:
['ReturnReference', 'SourceOrderReference', 'SourceLineReference', 'ReturnDate', 'Reason', 'Status', 'QtyReturned', 'RefundAmount', 'ReturnNotes', 'CreatedOn', 'CreatedByUser', 'ModifiedOn', 'ModifiedByUser']

Return Current pandas data types:
ReturnReference         object
SourceOrderReference    object
SourceLineReference     object
ReturnDate              object
Reason                  object
Status                  object
QtyReturned              int64
RefundAmount             int64
ReturnNotes             object
CreatedOn               object
CreatedByUser           object
ModifiedOn              object
ModifiedByUser          object
dtype: object

Return Missing values:
ReturnReference         0
SourceOrderReference    0
SourceLineReference     0
ReturnDate              0
Reason                  0
Status                  0
QtyReturned             0
RefundAmount            0
ReturnNotes             0
CreatedOn               0
Cre

,ReturnReference,SourceOrderReference,SourceLineReference,ReturnDate,Reason,Status,QtyReturned,RefundAmount,ReturnNotes,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,RE001,SO001,SOI021,2026-07-15,Defective,Completed,1,399,Refund approved,2026-07-15,admin,2026-07-16,admin
1,RE002,SO005,SOI005,2026-07-16,Changed mind,Completed,1,2999,Customer refund,2026-07-16,admin,2026-07-17,admin
2,RE003,SO013,SOI025,2026-07-22,Damaged,Approved,1,699,Replacement requested,2026-07-22,admin,2026-07-23,admin
3,RE004,SO015,SOI026,2026-07-23,Wrong item,Pending,1,699,Awaiting inspection,2026-07-23,admin,2026-07-23,admin


## 1.4 Data Profiling

Profile the untouched Return transaction dataset using the reusable project profiler.


In [4]:
# =========================================================
# 1.4 Data Profiling
# =========================================================

import sys
import importlib

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from profiler import data_profiler_v1 as profiler
importlib.reload(profiler)

print("Profiler loaded successfully:")
print(profiler.__file__)
print("\nProfiler configuration:")
print(profiler.DEFAULT_CONFIG)


Profiler loaded successfully:
c:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\profiler\data_profiler_v1.py

Profiler configuration:
{'date_detection_threshold': 0.8, 'numeric_detection_threshold': 0.8, 'email_detection_threshold': 0.8, 'phone_detection_threshold': 0.8, 'categorical_unique_ratio': 0.2, 'outlier_iqr_multiplier': 1.5, 'required_columns': [], 'unique_columns': [], 'non_negative_columns': []}


In [5]:
return_profile_results = profiler.profile_dataset(return_raw)
print("\nReturn profiling completed successfully.")
print("\nReturn profiler sections:")
print(list(return_profile_results.keys()))



Return profiling completed successfully.

Return profiler sections:
['field_types', 'detection_details', 'general', 'text', 'categorical', 'numeric', 'date', 'patterns', 'issues', 'configuration']


## 1.5 Review Profiling Results

Review the complete Return profiling output before making transformation decisions.


In [6]:
print("\n" + "=" * 70)
print("RETURN PROFILING RESULTS")
print("=" * 70)
for section_name, section_result in return_profile_results.items():
    print("\n" + "-" * 70)
    print(section_name.upper())
    print("-" * 70)
    print(section_result)



RETURN PROFILING RESULTS

----------------------------------------------------------------------
FIELD_TYPES
----------------------------------------------------------------------
ReturnReference         identifier
SourceOrderReference    identifier
SourceLineReference     identifier
ReturnDate                    date
Reason                        text
Status                        text
QtyReturned                numeric
RefundAmount               numeric
ReturnNotes                   text
CreatedOn                     date
CreatedByUser                 text
ModifiedOn                    date
ModifiedByUser                text
Name: DetectedFieldType, dtype: object

----------------------------------------------------------------------
DETECTION_DETAILS
----------------------------------------------------------------------
{'ReturnReference': {'DetectedType': 'identifier', 'Confidence': 'Medium', 'Evidence': 'Column name suggests identifier semantics'}, 'SourceOrderReference': {'Detec

## 1.6 Review Profiler Issues

Review the profiler's detected issues separately and classify them before transformation.


In [7]:
return_issues = return_profile_results["issues"]
print("\n" + "=" * 70)
print("RETURN PROFILER ISSUES")
print("=" * 70)
display(return_issues)



RETURN PROFILER ISSUES


,Column,IssueType,Severity,Count,Description


## 1.7 Independent Analyst Review

Perform the systematic independent analyst review required by the locked ETL workflow.


In [8]:
# =========================================================
# 1.7 Independent Analyst Review
# =========================================================

source_df = return_raw

print("1. DATASET STRUCTURE AND GRAIN")
print("=" * 80)
print(f"Rows: {source_df.shape[0]}")
print(f"Columns: {source_df.shape[1]}")
print("Working assumption: One row represents one return transaction.")

print("\n\n2. MISSING VALUES AND BLANK VALUES")
missing_values = source_df.isna().sum()
blank_values = pd.Series({c:(source_df[c].astype("string").str.strip().eq("").sum() if source_df[c].dtype == "object" else 0) for c in source_df.columns})
display(pd.DataFrame({"MissingValues":missing_values,"BlankValues":blank_values}))

print("\n\n3. EXACT DUPLICATE ROWS")
exact_duplicate_count = source_df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_count)
if exact_duplicate_count > 0: display(source_df[source_df.duplicated(keep=False)])

print("\n\n4. PRIMARY KEY CANDIDATE AND IDENTIFIER UNIQUENESS")
identifier_columns=["ReturnReference","SourceOrderReference","SourceLineReference"]
identifier_review=pd.DataFrame({"Column":identifier_columns,"NullCount":[source_df[c].isna().sum() for c in identifier_columns],"BlankCount":[source_df[c].astype("string").str.strip().eq("").sum() for c in identifier_columns],"DuplicateCount":[source_df[c].duplicated().sum() for c in identifier_columns],"UniqueValues":[source_df[c].nunique(dropna=True) for c in identifier_columns]})
display(identifier_review)

print("\n\n5. RETURN IDENTIFIER COLLISION REVIEW")
dups=source_df[source_df["ReturnReference"].duplicated(keep=False)]
print("Duplicate ReturnReference records:",len(dups))
if not dups.empty: display(dups)

print("\n\n6. PARENT / LINE RELATIONSHIP REVIEW")
print("Distinct SourceOrderReference values:",source_df["SourceOrderReference"].nunique())
print("Distinct SourceLineReference values:",source_df["SourceLineReference"].nunique())
display(source_df[["ReturnReference","SourceOrderReference","SourceLineReference"]])

print("\n\n7. LEADING AND TRAILING WHITESPACE REVIEW")
text_columns=source_df.select_dtypes(include="object").columns
whitespace_findings=[]
for column in text_columns:
    mask=source_df[column].astype("string") != source_df[column].astype("string").str.strip()
    count=mask.sum()
    if count>0:
        whitespace_findings.append({"Column":column,"WhitespaceRecordCount":count})
        display(source_df.loc[mask,["ReturnReference",column]])
if whitespace_findings: display(pd.DataFrame(whitespace_findings))
else: print("No leading or trailing whitespace found.")

print("\n\n8. TEXT / STATUS / REASON CONSISTENCY REVIEW")
for column in ["Reason","Status","ReturnNotes","CreatedByUser","ModifiedByUser"]:
    print(f"\n{column} values:")
    display(source_df[column].value_counts(dropna=False).sort_index().rename("RecordCount").to_frame())

print("\n\n9. RETURN QUANTITY AND REFUND AMOUNT VALIDITY REVIEW")
print("QtyReturned dtype:",source_df["QtyReturned"].dtype)
print("RefundAmount dtype:",source_df["RefundAmount"].dtype)
print("QtyReturned NULLs:",source_df["QtyReturned"].isna().sum())
print("QtyReturned <= 0:",(source_df["QtyReturned"]<=0).sum())
print("RefundAmount NULLs:",source_df["RefundAmount"].isna().sum())
print("RefundAmount < 0:",(source_df["RefundAmount"]<0).sum())
display(source_df[["ReturnReference","QtyReturned","RefundAmount"]])

print("\n\n10. DATE VALIDITY AND DATATYPE REVIEW")
for column in ["ReturnDate","CreatedOn","ModifiedOn"]:
    parsed=pd.to_datetime(source_df[column],errors="coerce")
    invalid=(parsed.isna() & source_df[column].notna()).sum()
    print(f"{column}: pandas dtype={source_df[column].dtype}, invalid populated dates={invalid}")

print("\n\n11. DATE RELATIONSHIP REVIEW")
return_date=pd.to_datetime(source_df["ReturnDate"],errors="coerce")
created=pd.to_datetime(source_df["CreatedOn"],errors="coerce")
modified=pd.to_datetime(source_df["ModifiedOn"],errors="coerce")
print("ModifiedOn before CreatedOn:",(modified.notna() & created.notna() & (modified<created)).sum())
print("ReturnDate before CreatedOn:",(return_date.notna() & created.notna() & (return_date<created)).sum())

print("\n\n12. BUSINESS RULE REVIEW")
print("Observed return statuses:",sorted(source_df["Status"].dropna().unique().tolist()))
print("Observed return reasons:",sorted(source_df["Reason"].dropna().unique().tolist()))
print("All ReturnReference values unique:",source_df["ReturnReference"].is_unique)
print("All QtyReturned values positive:",(source_df["QtyReturned"]>0).all())
print("All RefundAmount values non-negative:",(source_df["RefundAmount"]>=0).all())

print("\n\n13. SOURCE-TO-TARGET DATATYPE PREPARATION REVIEW")
datatype_review=pd.DataFrame([
["ReturnReference",str(source_df["ReturnReference"].dtype),"Text","Retain"],
["SourceOrderReference",str(source_df["SourceOrderReference"].dtype),"Text","Retain"],
["SourceLineReference",str(source_df["SourceLineReference"].dtype),"Text","Retain"],
["ReturnDate",str(source_df["ReturnDate"].dtype),"Datetime","Convert"],
["Reason",str(source_df["Reason"].dtype),"Text","Retain"],
["Status",str(source_df["Status"].dtype),"Text","Retain; no approved vocabulary replacement"],
["QtyReturned",str(source_df["QtyReturned"].dtype),"Integer","Retain"],
["RefundAmount",str(source_df["RefundAmount"].dtype),"Decimal","Prepare as numeric/decimal-compatible"],
["ReturnNotes",str(source_df["ReturnNotes"].dtype),"Text","Retain"],
["CreatedOn",str(source_df["CreatedOn"].dtype),"Datetime","Convert"],
["CreatedByUser",str(source_df["CreatedByUser"].dtype),"Text","Retain"],
["ModifiedOn",str(source_df["ModifiedOn"].dtype),"Datetime","Convert"],
["ModifiedByUser",str(source_df["ModifiedByUser"].dtype),"Text","Retain"]
],columns=["SourceColumn","CurrentPandasDtype","TargetType","Preparation"])
display(datatype_review)

print("\n\n14. REFERENTIAL READINESS REVIEW")
print("Return references Sales Order through SourceOrderReference.")
print("Return references Sales Order Item through SourceLineReference.")
print("Actual FK readiness is validated later against PostgreSQL.")


1. DATASET STRUCTURE AND GRAIN
Rows: 4
Columns: 13
Working assumption: One row represents one return transaction.


2. MISSING VALUES AND BLANK VALUES


,MissingValues,BlankValues
ReturnReference,0,0
SourceOrderReference,0,0
SourceLineReference,0,0
ReturnDate,0,0
Reason,0,0
Status,0,0
QtyReturned,0,0
RefundAmount,0,0
ReturnNotes,0,0
CreatedOn,0,0




3. EXACT DUPLICATE ROWS
Exact duplicate rows: 0


4. PRIMARY KEY CANDIDATE AND IDENTIFIER UNIQUENESS


,Column,NullCount,BlankCount,DuplicateCount,UniqueValues
0,ReturnReference,0,0,0,4
1,SourceOrderReference,0,0,0,4
2,SourceLineReference,0,0,0,4




5. RETURN IDENTIFIER COLLISION REVIEW
Duplicate ReturnReference records: 0


6. PARENT / LINE RELATIONSHIP REVIEW
Distinct SourceOrderReference values: 4
Distinct SourceLineReference values: 4


...



9. RETURN QUANTITY AND REFUND AMOUNT VALIDITY REVIEW
QtyReturned dtype: int64
RefundAmount dtype: int64
QtyReturned NULLs: 0
QtyReturned <= 0: 0
RefundAmount NULLs: 0
RefundAmount < 0: 0


,ReturnReference,QtyReturned,RefundAmount
0,RE001,1,399
1,RE002,1,2999
2,RE003,1,699
3,RE004,1,699




10. DATE VALIDITY AND DATATYPE REVIEW
ReturnDate: pandas dtype=object, invalid populated dates=0
CreatedOn: pandas dtype=object, invalid populated dates=0
ModifiedOn: pandas dtype=object, invalid populated dates=0


11. DATE RELATIONSHIP REVIEW
ModifiedOn before CreatedOn: 0
ReturnDate before CreatedOn: 0


12. BUSINESS RULE REVIEW
Observed return statuses: ['Approved', 'Completed', 'Pending']
Observed return reasons: ['Changed mind', 'Damaged', 'Defective', 'Wrong item']
All ReturnReference values unique: True
All QtyReturned values positive: True
All RefundAmount values non-negative: True


13. SOURCE-TO-TARGET DATATYPE PREPARATION REVIEW


,SourceColumn,CurrentPandasDtype,TargetType,Preparation
0,ReturnReference,object,Text,Retain
1,SourceOrderReference,object,Text,Retain
2,SourceLineReference,object,Text,Retain
3,ReturnDate,object,Datetime,Convert
4,Reason,object,Text,Retain
5,Status,object,Text,Retain; no approved vocabulary replacement
6,QtyReturned,int64,Integer,Retain
7,RefundAmount,int64,Decimal,Prepare as numeric/decimal-compatible
8,ReturnNotes,object,Text,Retain
9,CreatedOn,object,Datetime,Convert




14. REFERENTIAL READINESS REVIEW
Return references Sales Order through SourceOrderReference.
Return references Sales Order Item through SourceLineReference.
Actual FK readiness is validated later against PostgreSQL.


## 1.8 Findings and Transformation Decisions

Compare profiler findings, independent analyst findings, and the approved Return Data Dictionary/business rules. Only approved transformations proceed to Section 2.


In [9]:
transformation_decisions=pd.DataFrame([
["ReturnReference uniqueness",4,"Valid value — retain unchanged","Use as return_id","Approved primary identifier."],
["No exact duplicate rows",0,"Valid value — retain unchanged","Keep all 4 records","No duplicate records identified."],
["ReturnDate object dtype",4,"Safe deterministic transformation","Convert to datetime","Target return_date is Datetime and Optional."],
["CreatedOn object dtype",4,"Safe deterministic transformation","Convert to datetime","Target created_date is Datetime and Required."],
["ModifiedOn object dtype",4,"Safe deterministic transformation","Convert to datetime","Target updated_date is Datetime and Required."],
["RefundAmount integer source dtype",4,"Safe deterministic target preparation","Prepare as decimal/numeric-compatible value","Target refund_amount is Decimal and Optional."],
["QtyReturned positive integer values",4,"Valid value — retain unchanged","Keep values unchanged","Target return_quantity is Integer and Optional."],
["Status values Completed / Approved / Pending",4,"Valid source values — retain unchanged","Do not rename or collapse values","Current Return Data Dictionary does not enumerate permitted status values."],
["Reason values",4,"Valid value — retain unchanged","Keep values unchanged","No approved standardization rule requires replacement."],
["Notes and audit users",4,"Valid value — retain unchanged","Keep values unchanged","No cleaning requirement identified."],
["Sales Order / Sales Order Item references",4,"Requires relationship validation later","Validate against parent tables in PostgreSQL","Both are approved foreign keys."]
],columns=["Finding","AffectedRecords","Decision","ApprovedAction","Reason"])
display(transformation_decisions)
print("Return transformation decisions recorded. Status: READY FOR TRANSFORMATION")


,Finding,AffectedRecords,Decision,ApprovedAction,Reason
0,ReturnReference uniqueness,4,Valid value — retain unchanged,Use as return_id,Approved primary identifier.
1,No exact duplicate rows,0,Valid value — retain unchanged,Keep all 4 records,No duplicate records identified.
2,ReturnDate object dtype,4,Safe deterministic transformation,Convert to datetime,Target return_date is Datetime and Optional.
3,CreatedOn object dtype,4,Safe deterministic transformation,Convert to datetime,Target created_date is Datetime and Required.
4,ModifiedOn object dtype,4,Safe deterministic transformation,Convert to datetime,Target updated_date is Datetime and Required.
5,RefundAmount integer source dtype,4,Safe deterministic target preparation,Prepare as decimal/numeric-compatible value,Target refund_amount is Decimal and Optional.
6,QtyReturned positive integer values,4,Valid value — retain unchanged,Keep values unchanged,Target return_quantity is Integer and Optional.
7,Status values Completed / Approved / Pending,4,Valid source values — retain unchanged,Do not rename or collapse values,Current Return Data Dictionary does not enumerate permitted status values.
8,Reason values,4,Valid value — retain unchanged,Keep values unchanged,No approved standardization rule requires replacement.
9,Notes and audit users,4,Valid value — retain unchanged,Keep values unchanged,No cleaning requirement identified.


Return transformation decisions recorded. Status: READY FOR TRANSFORMATION


# =========================================================
# SECTION 2 — DATA TRANSFORMATION
# =========================================================

## 2.1 Create Clean Working Dataset

Create a separate working copy of the raw Return dataset before applying any approved transformations.

The raw Return dataset must remain unchanged throughout the ETL process.

All transformations will be applied only to the clean working dataset.

The Return source dataset will remain as a single working dataset during this step.

In [10]:
return_clean=return_raw.copy(deep=True)
print("Return Clean DataFrame created.")
print("Rows:",len(return_clean))
print("Columns:",len(return_clean.columns))


Return Clean DataFrame created.
Rows: 4
Columns: 13


## 2.2 Apply Approved Transformations

Apply only the approved Return transformations documented in Section 1.8.


In [11]:
for column in ["ReturnDate","CreatedOn","ModifiedOn"]:
    return_clean[column]=pd.to_datetime(return_clean[column],errors="raise")
return_clean["RefundAmount"]=pd.to_numeric(return_clean["RefundAmount"],errors="raise").astype("float64")
print("Approved Return transformations applied.")


Approved Return transformations applied.


## 3.1 Transformation Validation

Validate every approved Return transformation from Section 1.8.

Checks include datetime conversion, RefundAmount numeric/decimal compatibility, preservation of Status values, preservation of approved Return values, and approved Return business rules.

In [12]:
datetime_valid=all(pd.api.types.is_datetime64_any_dtype(return_clean[c]) for c in ["ReturnDate","CreatedOn","ModifiedOn"])
refund_numeric_valid=pd.api.types.is_float_dtype(return_clean["RefundAmount"])
status_values_preserved=return_clean["Status"].tolist()==return_raw["Status"].tolist()
print("Return Transformation Validation")
print("Datetime fields valid:",datetime_valid)
print("RefundAmount numeric/decimal-compatible:",refund_numeric_valid)
print("Status values preserved:",status_values_preserved)
transformation_validation_passed=all([datetime_valid,refund_numeric_valid,status_values_preserved])
print("Overall transformation validation passed:",transformation_validation_passed)


Return Transformation Validation
Datetime fields valid: True
RefundAmount numeric/decimal-compatible: True
Status values preserved: True
Overall transformation validation passed: True


## 3.2 Validate Data Preservation

Validate that the approved Return transformations did not unintentionally change unrelated values or remove records.


In [13]:
preserved_columns=["ReturnReference","SourceOrderReference","SourceLineReference","Reason","Status","QtyReturned","ReturnNotes","CreatedByUser","ModifiedByUser"]
row_count_preserved=len(return_clean)==len(return_raw)
column_structure_preserved=return_clean.columns.tolist()==return_raw.columns.tolist()
preservation_results={c:return_clean[c].astype("string").equals(return_raw[c].astype("string")) for c in preserved_columns}
preserved_values_valid=all(preservation_results.values())
return_ids_preserved=return_clean["ReturnReference"].tolist()==return_raw["ReturnReference"].tolist()
refund_values_preserved=return_clean["RefundAmount"].astype(float).tolist()==return_raw["RefundAmount"].astype(float).tolist()
data_preservation_passed=all([row_count_preserved,column_structure_preserved,preserved_values_valid,return_ids_preserved,refund_values_preserved])
display(pd.Series(preservation_results,name="Preserved"))
print("Row count preserved:",row_count_preserved)
print("Column structure preserved:",column_structure_preserved)
print("Return IDs preserved:",return_ids_preserved)
print("Refund values preserved:",refund_values_preserved)
print("Unchanged values preserved:",preserved_values_valid)
print("Overall data preservation validation passed:",data_preservation_passed)


ReturnReference         True
SourceOrderReference    True
SourceLineReference     True
Reason                  True
Status                  True
QtyReturned             True
ReturnNotes             True
CreatedByUser           True
ModifiedByUser          True
Name: Preserved, dtype: bool

Row count preserved: True
Column structure preserved: True
Return IDs preserved: True
Refund values preserved: True
Unchanged values preserved: True
Overall data preservation validation passed: True


# =========================================================
# SECTION 4 — CLEAN CSV
# =========================================================


In [14]:
# =========================================================
# 4.1 Export Clean Return Dataset
# =========================================================

return_clean_dir=Path(r"C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\11_Returns_System\clean")
return_clean_dir.mkdir(parents=True,exist_ok=True)
return_clean_path=return_clean_dir / "returns_transactions_clean.csv"
return_clean.to_csv(return_clean_path,index=False)
print("Clean Return CSV exported successfully.")
print("Path:",return_clean_path)


Clean Return CSV exported successfully.
Path: C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\11_Returns_System\clean\returns_transactions_clean.csv


## 4.2 Verify Exported Clean Return CSV

Reload the exported Clean CSV and verify row count and column structure against the validated Clean DataFrame.


In [15]:
return_clean_csv=pd.read_csv(return_clean_path,parse_dates=["ReturnDate","CreatedOn","ModifiedOn"])
row_count_match=len(return_clean_csv)==len(return_clean)
column_structure_match=return_clean_csv.columns.tolist()==return_clean.columns.tolist()
print("Return Clean CSV read-back successful.")
print("Rows:",len(return_clean_csv))
print("Columns:",len(return_clean_csv.columns))
print("Row count matches:",row_count_match)
print("Column structure matches:",column_structure_match)
display(return_clean_csv.head())


Return Clean CSV read-back successful.
Rows: 4
Columns: 13
Row count matches: True
Column structure matches: True


,ReturnReference,SourceOrderReference,SourceLineReference,ReturnDate,Reason,Status,QtyReturned,RefundAmount,ReturnNotes,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,RE001,SO001,SOI021,2026-07-15,Defective,Completed,1,399.0,Refund approved,2026-07-15,admin,2026-07-16,admin
1,RE002,SO005,SOI005,2026-07-16,Changed mind,Completed,1,2999.0,Customer refund,2026-07-16,admin,2026-07-17,admin
2,RE003,SO013,SOI025,2026-07-22,Damaged,Approved,1,699.0,Replacement requested,2026-07-22,admin,2026-07-23,admin
3,RE004,SO015,SOI026,2026-07-23,Wrong item,Pending,1,699.0,Awaiting inspection,2026-07-23,admin,2026-07-23,admin


# =========================================================
# SECTION 5 — DATABASE READY
# =========================================================


In [16]:
# =========================================================
# 5.1 Define Source → Target Mapping
# =========================================================

return_mapping=pd.DataFrame([
["ReturnReference","return_id","TEXT","Required","PK",""],
["SourceOrderReference","sales_order_id","TEXT","Required","FK","commerce.sales_order.sales_order_id"],
["SourceLineReference","sales_order_item_id","TEXT","Required","FK","commerce.sales_order_item.sales_order_item_id"],
["ReturnDate","return_date","TIMESTAMP","Optional","",""],
["Reason","return_reason","TEXT","Required","",""],
["Status","return_status","TEXT","Required","","Retain observed source values; no permitted vocabulary supplied"],
["QtyReturned","return_quantity","INTEGER","Optional","",""],
["RefundAmount","refund_amount","NUMERIC","Optional","",""],
["ReturnNotes","notes","TEXT","Optional","",""],
["CreatedOn","created_date","TIMESTAMP","Required","",""],
["CreatedByUser","created_by","TEXT","Required","",""],
["ModifiedOn","updated_date","TIMESTAMP","Required","",""],
["ModifiedByUser","updated_by","TEXT","Required","",""]
],columns=["Source Column","Target Column","Target Datatype","Required","Key Role","Reference / Rule"])
display(return_mapping)


,Source Column,Target Column,Target Datatype,Required,Key Role,Reference / Rule
0,ReturnReference,return_id,TEXT,Required,PK,
1,SourceOrderReference,sales_order_id,TEXT,Required,FK,commerce.sales_order.sales_order_id
2,SourceLineReference,sales_order_item_id,TEXT,Required,FK,commerce.sales_order_item.sales_order_item_id
3,ReturnDate,return_date,TIMESTAMP,Optional,,
4,Reason,return_reason,TEXT,Required,,
5,Status,return_status,TEXT,Required,,Retain observed source values; no permitted vocabulary supplied
6,QtyReturned,return_quantity,INTEGER,Optional,,
7,RefundAmount,refund_amount,NUMERIC,Optional,,
8,ReturnNotes,notes,TEXT,Optional,,
9,CreatedOn,created_date,TIMESTAMP,Required,,


## 5.2 Create Database-Ready Return Dataset

Create the Database-Ready Return dataset from the verified exported Clean CSV, not directly from the in-memory Clean DataFrame.


In [17]:
return_db_source=pd.read_csv(return_clean_path,parse_dates=["ReturnDate","CreatedOn","ModifiedOn"])
return_db_ready=return_db_source.rename(columns={"ReturnReference":"return_id","SourceOrderReference":"sales_order_id","SourceLineReference":"sales_order_item_id","ReturnDate":"return_date","Reason":"return_reason","Status":"return_status","QtyReturned":"return_quantity","RefundAmount":"refund_amount","ReturnNotes":"notes","CreatedOn":"created_date","CreatedByUser":"created_by","ModifiedOn":"updated_date","ModifiedByUser":"updated_by"})[["return_id","sales_order_id","sales_order_item_id","return_date","return_reason","return_status","return_quantity","refund_amount","notes","created_date","created_by","updated_date","updated_by"]].copy()
return_db_ready["refund_amount"]=return_db_ready["refund_amount"].apply(lambda value: Decimal(str(value)) if pd.notna(value) else None)
print("Return Database-Ready dataset created from exported Clean CSV.")
print("Rows:",len(return_db_ready))
print("Columns:",len(return_db_ready.columns))
display(return_db_ready.head())


Return Database-Ready dataset created from exported Clean CSV.
Rows: 4
Columns: 13


,return_id,sales_order_id,sales_order_item_id,return_date,return_reason,return_status,return_quantity,refund_amount,notes,created_date,created_by,updated_date,updated_by
0,RE001,SO001,SOI021,2026-07-15,Defective,Completed,1,399.0,Refund approved,2026-07-15,admin,2026-07-16,admin
1,RE002,SO005,SOI005,2026-07-16,Changed mind,Completed,1,2999.0,Customer refund,2026-07-16,admin,2026-07-17,admin
2,RE003,SO013,SOI025,2026-07-22,Damaged,Approved,1,699.0,Replacement requested,2026-07-22,admin,2026-07-23,admin
3,RE004,SO015,SOI026,2026-07-23,Wrong item,Pending,1,699.0,Awaiting inspection,2026-07-23,admin,2026-07-23,admin


## 5.3 Database-Ready Validation

Validate the Return Database-Ready dataset against the approved target columns, required fields, primary key, datatypes, mapping completeness, and preserved source values.


In [18]:
expected_target_columns=["return_id","sales_order_id","sales_order_item_id","return_date","return_reason","return_status","return_quantity","refund_amount","notes","created_date","created_by","updated_date","updated_by"]
target_columns_match=return_db_ready.columns.tolist()==expected_target_columns
row_count_match=len(return_db_ready)==len(return_clean_csv)
required_fields=["return_id","sales_order_id","sales_order_item_id","return_reason","return_status","created_date","created_by","updated_date","updated_by"]
required_null_counts=return_db_ready[required_fields].isnull().sum()
required_fields_valid=required_null_counts.sum()==0
duplicate_primary_keys=return_db_ready["return_id"].duplicated().sum()
null_primary_keys=return_db_ready["return_id"].isnull().sum()
primary_key_valid=duplicate_primary_keys==0 and null_primary_keys==0
datetime_fields=["return_date","created_date","updated_date"]
datetime_fields_valid=all(pd.api.types.is_datetime64_any_dtype(return_db_ready[c]) for c in datetime_fields)
refund_decimal_valid=return_db_ready["refund_amount"].dropna().map(lambda value:isinstance(value,Decimal)).all()
mapping_complete=set(return_mapping["Source Column"])==set(return_clean_csv.columns)
status_values_preserved=set(return_db_ready["return_status"].dropna().unique())==set(return_clean_csv["Status"].dropna().unique())
display(required_null_counts)
print("Target columns match:",target_columns_match)
print("Row count match:",row_count_match)
print("Required fields valid:",required_fields_valid)
print("Primary key valid:",primary_key_valid)
print("Datetime fields valid:",datetime_fields_valid)
print("RefundAmount Decimal-ready:",refund_decimal_valid)
print("Status values preserved:",status_values_preserved)
print("Mapping complete:",mapping_complete)
database_ready_validation_passed=all([target_columns_match,row_count_match,required_fields_valid,primary_key_valid,datetime_fields_valid,refund_decimal_valid,status_values_preserved,mapping_complete])
print("Return Database-Ready validation passed:",database_ready_validation_passed)


return_id              0
sales_order_id         0
sales_order_item_id    0
return_reason          0
return_status          0
created_date           0
created_by             0
updated_date           0
updated_by             0
dtype: int64

Target columns match: True
Row count match: True
Required fields valid: True
Primary key valid: True
Datetime fields valid: True
RefundAmount Decimal-ready: True
Status values preserved: True
Mapping complete: True
Return Database-Ready validation passed: True


# =========================================================
# SECTION 6 — PostgreSQL
# =========================================================


## 6.1 Connect

Establish a PostgreSQL connection for the Return ETL process.


In [19]:
import psycopg2
from getpass import getpass
DB_HOST="localhost"
DB_PORT="5432"
DB_NAME="lj_dev_commerce"
DB_USER="postgres"
DB_PASSWORD=getpass("Enter PostgreSQL password: ")
try:
    conn=psycopg2.connect(host=DB_HOST,port=DB_PORT,dbname=DB_NAME,user=DB_USER,password=DB_PASSWORD)
    cursor=conn.cursor()
    print("PostgreSQL connection successful.")
    print("Database:",DB_NAME)
    print("Host:",DB_HOST)
    print("Port:",DB_PORT)
except Exception as e:
    print("PostgreSQL connection failed.")
    print("Error:",e)


PostgreSQL connection successful.
Database: lj_dev_commerce
Host: localhost
Port: 5432


## 6.2 Prepare / Create Target Table

Check whether the approved `commerce.return` target table already exists. If it exists, do not recreate it.


In [20]:
target_schema="commerce"
target_table="return"
cursor.execute("""SELECT EXISTS (SELECT 1 FROM information_schema.tables WHERE table_schema=%s AND table_name=%s);""",(target_schema,target_table))
table_exists=cursor.fetchone()[0]
print("Return Target Table Check")
print("="*80)
print("Schema:",target_schema)
print("Table:",target_table)
print("Table exists:",table_exists)
if table_exists:
    print("\nTarget table already exists.")
    print("The table will not be recreated.")
else:
    print("\nTarget table does not exist.")
    print("The approved database design will be used before creating it.")


Return Target Table Check
Schema: commerce
Table: return
Table exists: False

Target table does not exist.
The approved database design will be used before creating it.


## 6.2.1 Create Return Table

Create `commerce.return` only when the target table does not already exist, using the approved Return Data Dictionary and foreign-key relationships.


In [21]:
if not table_exists:
    create_return_table_query="""
    CREATE TABLE commerce.return (
        return_id TEXT PRIMARY KEY,
        sales_order_id TEXT NOT NULL,
        sales_order_item_id TEXT NOT NULL,
        return_date TIMESTAMP,
        return_reason TEXT NOT NULL,
        return_status TEXT NOT NULL,
        return_quantity INTEGER,
        refund_amount NUMERIC,
        notes TEXT,
        created_date TIMESTAMP NOT NULL,
        created_by TEXT NOT NULL,
        updated_date TIMESTAMP NOT NULL,
        updated_by TEXT NOT NULL,
        FOREIGN KEY (sales_order_id) REFERENCES commerce.sales_order(sales_order_id),
        FOREIGN KEY (sales_order_item_id) REFERENCES commerce.sales_order_item(sales_order_item_id)
    );
    """
    try:
        cursor.execute(create_return_table_query)
        conn.commit()
        print("Return table created successfully.")
    except Exception as e:
        conn.rollback()
        print("Return table creation failed.")
        print("Error:",e)
else:
    print("Return table already exists. Creation skipped.")


Return table created successfully.


## 6.3 Verify Return Table Structure

Verify the actual PostgreSQL structure of `commerce.return` against the approved target design.


In [22]:
verify_return_table_query="""SELECT ordinal_position,column_name,data_type,is_nullable,column_default FROM information_schema.columns WHERE table_schema='commerce' AND table_name='return' ORDER BY ordinal_position;"""
try:
    cursor.execute(verify_return_table_query)
    table_structure=cursor.fetchall()
    print("Return Table Structure")
    print("="*80)
    print(f"{'Position':<10}{'Column Name':<25}{'Data Type':<25}{'Nullable':<12}{'Default'}")
    print("-"*100)
    for row in table_structure:
        print(f"{row[0]:<10}{row[1]:<25}{row[2]:<25}{row[3]:<12}{row[4]}")
except Exception as e:
    print("Return table structure verification failed.")
    print("Error:",e)


Return Table Structure
Position  Column Name              Data Type                Nullable    Default
----------------------------------------------------------------------------------------------------
1         return_id                text                     NO          None
2         sales_order_id           text                     NO          None
3         sales_order_item_id      text                     NO          None
4         return_date              timestamp without time zoneYES         None
5         return_reason            text                     NO          None
6         return_status            text                     NO          None
7         return_quantity          integer                  YES         None
8         refund_amount            numeric                  YES         None
9         notes                    text                     YES         None
10        created_date             timestamp without time zoneNO          None
11        created_by  

## 6.4 Verify Keys, Relationships & Load Dependencies

Verify the Return primary key, both approved foreign keys, and the required parent records in Sales Order and Sales Order Item.


In [23]:
print("Return Keys, Relationships & Load Dependency Review")
print("="*80)
cursor.execute("""SELECT tc.constraint_name,kcu.column_name FROM information_schema.table_constraints AS tc JOIN information_schema.key_column_usage AS kcu ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema WHERE tc.table_schema='commerce' AND tc.table_name='return' AND tc.constraint_type='PRIMARY KEY';""")
primary_keys=cursor.fetchall()
primary_key_valid=len(primary_keys)==1 and primary_keys[0][1]=="return_id"
print("Primary Key:",primary_keys)
print("Primary Key validation passed:",primary_key_valid)
cursor.execute("""SELECT tc.constraint_name,kcu.column_name,ccu.table_schema,ccu.table_name,ccu.column_name FROM information_schema.table_constraints AS tc JOIN information_schema.key_column_usage AS kcu ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema JOIN information_schema.constraint_column_usage AS ccu ON ccu.constraint_name=tc.constraint_name AND ccu.table_schema=tc.table_schema WHERE tc.table_schema='commerce' AND tc.table_name='return' AND tc.constraint_type='FOREIGN KEY' ORDER BY kcu.column_name;""")
foreign_keys=cursor.fetchall()
expected_foreign_keys={("sales_order_id","sales_order","sales_order_id"),("sales_order_item_id","sales_order_item","sales_order_item_id")}
actual_foreign_keys={(r[1],r[3],r[4]) for r in foreign_keys}
foreign_keys_valid=actual_foreign_keys==expected_foreign_keys
print("Foreign Keys:",foreign_keys)
print("Foreign Key validation passed:",foreign_keys_valid)
sales_order_ids=return_db_ready["sales_order_id"].dropna().unique().tolist()
placeholders=",".join(["%s"]*len(sales_order_ids))
cursor.execute(f"SELECT sales_order_id FROM commerce.sales_order WHERE sales_order_id IN ({placeholders});",sales_order_ids)
existing_sales_order_ids={r[0] for r in cursor.fetchall()}
missing_sales_order_ids=set(sales_order_ids)-existing_sales_order_ids
sales_order_dependency_valid=len(missing_sales_order_ids)==0
sales_order_item_ids=return_db_ready["sales_order_item_id"].dropna().unique().tolist()
placeholders=",".join(["%s"]*len(sales_order_item_ids))
cursor.execute(f"SELECT sales_order_item_id FROM commerce.sales_order_item WHERE sales_order_item_id IN ({placeholders});",sales_order_item_ids)
existing_sales_order_item_ids={r[0] for r in cursor.fetchall()}
missing_sales_order_item_ids=set(sales_order_item_ids)-existing_sales_order_item_ids
sales_order_item_dependency_valid=len(missing_sales_order_item_ids)==0
print("Missing Sales Order IDs:",missing_sales_order_ids)
print("Sales Order dependency ready:",sales_order_dependency_valid)
print("Missing Sales Order Item IDs:",missing_sales_order_item_ids)
print("Sales Order Item dependency ready:",sales_order_item_dependency_valid)
load_dependency_ready=all([primary_key_valid,foreign_keys_valid,sales_order_dependency_valid,sales_order_item_dependency_valid])
print("Return is ready for data loading:",load_dependency_ready)


Return Keys, Relationships & Load Dependency Review
Primary Key: [('return_pkey', 'return_id')]
Primary Key validation passed: True
Foreign Keys: [('return_sales_order_id_fkey', 'sales_order_id', 'commerce', 'sales_order', 'sales_order_id'), ('return_sales_order_item_id_fkey', 'sales_order_item_id', 'commerce', 'sales_order_item', 'sales_order_item_id')]
Foreign Key validation passed: True
Missing Sales Order IDs: set()
Sales Order dependency ready: True
Missing Sales Order Item IDs: set()
Sales Order Item dependency ready: True
Return is ready for data loading: True


## 6.5 Insert Return Records

Load the validated Return Database-Ready dataset only after target structure and dependency checks pass. Insertion is protected against accidental duplicate loading.


In [24]:
print("Return Record Loading")
print("="*80)
cursor.execute("SELECT COUNT(*) FROM commerce.return;")
existing_record_count=cursor.fetchone()[0]
target_table_empty=existing_record_count==0
db_ready_row_count=len(return_db_ready)
print("Existing records:",existing_record_count)
print("Target table is empty:",target_table_empty)
print("Database-ready records:",db_ready_row_count)
insert_return_query="""INSERT INTO commerce.return (return_id,sales_order_id,sales_order_item_id,return_date,return_reason,return_status,return_quantity,refund_amount,notes,created_date,created_by,updated_date,updated_by) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s);"""
try:
    if not target_table_empty: raise ValueError("Target table is not empty. Insertion stopped to prevent accidental duplicate loading.")
    records_to_insert=[tuple(row) for row in return_db_ready.itertuples(index=False,name=None)]
    if len(records_to_insert)!=db_ready_row_count: raise ValueError("Database-ready record count changed before insertion.")
    cursor.executemany(insert_return_query,records_to_insert)
    conn.commit()
    print("Return records inserted:",len(records_to_insert))
    print("Transaction committed successfully.")
except Exception as e:
    conn.rollback()
    print("Return record insertion failed.")
    print("Transaction rolled back.")
    print("Error:",e)


Return Record Loading
Existing records: 0
Target table is empty: True
Database-ready records: 4
Return records inserted: 4
Transaction committed successfully.


## 6.6 Validate Return Row Count

Compare the Database-Ready Return row count with the PostgreSQL target row count.


In [25]:
db_ready_row_count=len(return_db_ready)
cursor.execute("SELECT COUNT(*) FROM commerce.return;")
postgresql_row_count=cursor.fetchone()[0]
row_count_match=db_ready_row_count==postgresql_row_count
print("Return Row Count Validation")
print("="*80)
print("\nDatabase-Ready Dataset Row Count:",db_ready_row_count)
print("PostgreSQL Table Row Count:",postgresql_row_count)
print("Row counts match:",row_count_match)
print("Return row count validation passed." if row_count_match else "Return row count validation failed.")


Return Row Count Validation

Database-Ready Dataset Row Count: 4
PostgreSQL Table Row Count: 4
Row counts match: True
Return row count validation passed.


## 6.7 Retrieve Return Records

Retrieve the loaded Return records from PostgreSQL for direct review.


In [26]:
retrieve_return_query="""SELECT return_id,sales_order_id,sales_order_item_id,return_date,return_reason,return_status,return_quantity,refund_amount,notes,created_date,created_by,updated_date,updated_by FROM commerce.return ORDER BY return_id;"""
try:
    cursor.execute(retrieve_return_query)
    return_records=cursor.fetchall()
    column_names=[description[0] for description in cursor.description]
    return_postgresql=pd.DataFrame(return_records,columns=column_names)
    for column in ["return_date","created_date","updated_date"]:
        return_postgresql[column]=pd.to_datetime(return_postgresql[column],errors="coerce")
    print("Return Records Retrieved from PostgreSQL")
    print("Records retrieved:",len(return_records))
    display(return_postgresql)
except Exception as e:
    print("Return record retrieval failed.")
    print("Error:",e)


Return Records Retrieved from PostgreSQL
Records retrieved: 4


,return_id,sales_order_id,sales_order_item_id,return_date,return_reason,return_status,return_quantity,refund_amount,notes,created_date,created_by,updated_date,updated_by
0,RE001,SO001,SOI021,2026-07-15,Defective,Completed,1,399.0,Refund approved,2026-07-15,admin,2026-07-16,admin
1,RE002,SO005,SOI005,2026-07-16,Changed mind,Completed,1,2999.0,Customer refund,2026-07-16,admin,2026-07-17,admin
2,RE003,SO013,SOI025,2026-07-22,Damaged,Approved,1,699.0,Replacement requested,2026-07-22,admin,2026-07-23,admin
3,RE004,SO015,SOI026,2026-07-23,Wrong item,Pending,1,699.0,Awaiting inspection,2026-07-23,admin,2026-07-23,admin


## 6.8 Source → Database Reconciliation

Compare the Return Database-Ready dataset against the PostgreSQL target for row count, columns, primary keys, and record-level values.


In [27]:
return_source_reconciliation=return_db_ready.copy().sort_values("return_id").reset_index(drop=True)
return_postgresql_reconciliation=return_postgresql.copy().sort_values("return_id").reset_index(drop=True)
for column in ["return_date","created_date","updated_date"]:
    return_source_reconciliation[column]=pd.to_datetime(return_source_reconciliation[column],errors="coerce")
    return_postgresql_reconciliation[column]=pd.to_datetime(return_postgresql_reconciliation[column],errors="coerce")
source_row_count=len(return_source_reconciliation)
database_row_count=len(return_postgresql_reconciliation)
row_count_match=source_row_count==database_row_count
source_columns=return_source_reconciliation.columns.tolist()
database_columns=return_postgresql_reconciliation.columns.tolist()
column_match=source_columns==database_columns
source_return_ids=set(return_source_reconciliation["return_id"])
database_return_ids=set(return_postgresql_reconciliation["return_id"])
primary_key_match=source_return_ids==database_return_ids
return_source_reconciliation["refund_amount"]=return_source_reconciliation["refund_amount"].apply(lambda v:Decimal(str(v)) if pd.notna(v) else None)
return_postgresql_reconciliation["refund_amount"]=return_postgresql_reconciliation["refund_amount"].apply(lambda v:Decimal(str(v)) if pd.notna(v) else None)
value_match=return_source_reconciliation.equals(return_postgresql_reconciliation)
reconciliation_passed=all([row_count_match,column_match,primary_key_match,value_match])
print("Return Source → Database Reconciliation")
print("Row counts match:",row_count_match)
print("Columns and order match:",column_match)
print("Return IDs match:",primary_key_match)
print("All record values match:",value_match)
print("Source → Database reconciliation passed:",reconciliation_passed)


Return Source → Database Reconciliation
Row counts match: True
Columns and order match: True
Return IDs match: True
All record values match: True
Source → Database reconciliation passed: True


## 6.9 Return Database Integrity Validation

Perform final Return integrity checks covering required fields, primary key uniqueness, FK-related fields, quantity/refund validity, and audit date relationships.


In [28]:
integrity_query="""SELECT COUNT(*) AS total_rows, COUNT(DISTINCT return_id) AS distinct_return_ids, COUNT(*) FILTER (WHERE return_id IS NULL) AS null_return_ids, COUNT(*) FILTER (WHERE sales_order_id IS NULL) AS null_sales_order_ids, COUNT(*) FILTER (WHERE sales_order_item_id IS NULL) AS null_sales_order_item_ids, COUNT(*) FILTER (WHERE return_reason IS NULL) AS null_return_reasons, COUNT(*) FILTER (WHERE return_status IS NULL) AS null_return_statuses, COUNT(*) FILTER (WHERE created_date IS NULL) AS null_created_dates, COUNT(*) FILTER (WHERE created_by IS NULL) AS null_created_by, COUNT(*) FILTER (WHERE updated_date IS NULL) AS null_updated_dates, COUNT(*) FILTER (WHERE updated_by IS NULL) AS null_updated_by, COUNT(*) FILTER (WHERE return_quantity IS NOT NULL AND return_quantity <= 0) AS invalid_return_quantities, COUNT(*) FILTER (WHERE refund_amount IS NOT NULL AND refund_amount < 0) AS negative_refund_amounts, COUNT(*) FILTER (WHERE updated_date < created_date) AS invalid_audit_date_order, COUNT(*) FILTER (WHERE return_date IS NOT NULL AND return_date < created_date) AS return_before_created FROM commerce.return;"""
try:
    cursor.execute(integrity_query)
    r=cursor.fetchone()
    print("Return Database Integrity Validation")
    print("="*80)
    labels=["Total rows","Distinct return IDs","NULL return IDs","NULL sales order IDs","NULL sales order item IDs","NULL return reasons","NULL return statuses","NULL created dates","NULL created by","NULL updated dates","NULL updated by","Invalid return quantities","Negative refund amounts","Updated before created","Return before created"]
    for label,value in zip(labels,r): print(f"{label}: {value}")
    integrity_passed=(r[0]==r[1] and all(value==0 for value in r[2:]))
    print("\nReturn database integrity passed:",integrity_passed)
except Exception as e:
    print("Return database integrity validation failed.")
    print("Error:",e)


Return Database Integrity Validation
Total rows: 4
Distinct return IDs: 4
NULL return IDs: 0
NULL sales order IDs: 0
NULL sales order item IDs: 0
NULL return reasons: 0
NULL return statuses: 0
NULL created dates: 0
NULL created by: 0
NULL updated dates: 0
NULL updated by: 0
Invalid return quantities: 0
Negative refund amounts: 0
Updated before created: 0
Return before created: 0

Return database integrity passed: True


## 6.10 Close PostgreSQL Connection

Close PostgreSQL resources cleanly after all Return loading and validation steps are complete.


In [29]:
try:
    if cursor is not None and not cursor.closed: cursor.close()
    if conn is not None and conn.closed == 0: conn.close()
    print("PostgreSQL resources closed successfully.")
    print("Cursor closed:",cursor.closed)
    print("Connection closed:",conn.closed != 0)
except Exception as e:
    print("PostgreSQL resource closure failed.")
    print("Error:",e)


PostgreSQL resources closed successfully.
Cursor closed: True
Connection closed: True
